# VAZHI SFT v7.0 — Gemma 3 1B-it Tamil Fine-Tuning

**Major pivot:** First training on Gemma 3 1B-it after Model Comparison v1 proved it
outperforms 20 Qwen3-0.6B training attempts with zero fine-tuning.

```
Lineage: google/gemma-3-1b-it (vanilla) → SFT v7.0 (this notebook)
```

**Why Gemma 3 1B-it:**
- Already has genuine Tamil from 2T-token multilingual pretraining (140+ languages, 262K vocab)
- Zero fine-tuning produces real Tamil words, relevant answers, structured output
- Q4_K_M GGUF = 0.60 GB (well within <1GB mobile limit)
- No DAPT needed — dramatically simpler pipeline

**SFT v7.0 goal:** Ground VAZHI in Tamil spiritual culture (Kural, Sadhguru, Siddhars)
while teaching domain knowledge, VAZHI identity/mission, and mobile-appropriate response style.

**Issues to fix from benchmark:**
- Response degeneration at end (garbage tokens)
- Factual hallucinations (Thanjavur as capital instead of Chennai)
- Identity: says "VAZI" not "VAZHI"
- Navarasa-style catastrophic forgetting risk — use conservative LoRA

**Dataset:** `CryptoYogi/vazhi-tamil-sft-v7_0` (4,172 samples: 3,754 train + 418 eval)
- Rebalanced for Gemma 3, spiritual wisdom preserved as core identity
- 61 mission pairs: VAZHI acronym, open source philosophy, offline/no-inference-cost advantage,
  user feedback, knowledge packs, sponsorship/scholarship model, community contribution
- Format: raw instruction/output — Gemma template applied in this notebook

**Token distribution (v5.3 → v7.0):**

| Bucket | v5.3 %Tokens | v7.0 %Tokens | Purpose |
|--------|-------------|-------------|---------|
| Vazhi packs (domain) | 18.8% | 51.8% | Practical knowledge (6 domains) |
| Sadhguru (spiritual) | 78.3% | 39.2% | Tamil spiritual wisdom (truncated to 200 words) |
| Thirukkural | 0.9% | 2.5% | Classical wisdom |
| Conversational | 0.7% | 1.8% | Greetings, identity, thanks |
| Mission | — | 1.2% | VAZHI philosophy, offline, feedback, sponsorship |
| Behavior | 0.4% | 1.1% | Domain awareness, graceful limits, tone |
| Handcrafted | 0.4% | 1.0% | Safety refusals |
| Safety | 0.5% | 0.8% | Tamil-language refusal patterns |
| Corrections | — | 0.3% | Gemma 3 factual error fixes |
| General | 0.1% | 0.2% | General Tamil Q&A |
| **Identity+behavior** | **1.5%** | **5.5%** | Combined identity bucket |
| **Avg answer** | **125 words** | **47 words** | Mobile-appropriate |

**Runtime:** Colab Pro GPU (L4 recommended, A100 ideal). ~1B params = ~2GB fp16.

In [1]:
# Cell 1 — Dependencies
!pip install -q -U \
  "transformers>=4.50.0,<5.0.0" \
  "trl>=0.20.0" \
  "datasets>=2.21.0" \
  "peft>=0.13.0" \
  "accelerate>=0.34.0" \
  "huggingface_hub>=0.24.7"

import torch
print(f"\u2705 Dependencies installed")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"   VRAM: {vram / 1024**3:.0f} GB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.3 MB/s eta 0:00:00
✅ Dependencies installed
   PyTorch: 2.9.0+cu128
   CUDA: True
   GPU: NVIDIA L4
   VRAM: 22 GB


In [2]:
# Cell 2 — Configuration

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import re
import random
import gc
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset, Dataset
from huggingface_hub import login, HfApi

from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# === ENVIRONMENT DETECTION ===
IS_KAGGLE = os.path.exists("/kaggle/working")
WORK_DIR = "/kaggle/working" if IS_KAGGLE else "/content"
ENV_NAME = "Kaggle" if IS_KAGGLE else "Colab"

# === KEY CONFIG ===
BASE_MODEL = "google/gemma-3-1b-it"               # Gemma 3 1B instruct (native Tamil)
SFT_DATASET = "CryptoYogi/vazhi-tamil-sft-v7_0"   # v7.0: rebalanced for Gemma 3
OUTPUT_MODEL = "CryptoYogi/vazhi-v7_0"             # Final VAZHI model
ADAPTER_REPO = "CryptoYogi/vazhi-v7_0-lora"        # Adapter backup

# Training config — CONSERVATIVE to avoid Navarasa-style catastrophic forgetting
# Navarasa 2.0 SFT destroyed Gemma's instruction-following. We use:
# - Low LR (1e-5, proven safe in v5.0/v5.1a)
# - Small LoRA rank (r=8, fewer trainable params to reduce forgetting risk)
# - Only attention modules (q_proj, v_proj — minimal intervention)
# - 1 epoch only (evaluate before committing to more)
LEARNING_RATE = 1e-5       # Conservative — Navarasa used aggressive LR and broke the model
NUM_EPOCHS = 1             # Start with 1, eval before deciding on epoch 2
MAX_LENGTH = 2048          # Comfortable for v7.0 dataset (avg 37 words/answer)
LORA_R = 8                 # Conservative — less risk of catastrophic forgetting than r=16
LORA_ALPHA = 16            # Standard 2x ratio
LORA_TARGETS = ["q_proj", "v_proj"]  # Minimal intervention — attention only
BATCH_SIZE = 4             # Per-device (adjust: 2=T4, 4=L4, 8=A100)
GRADIENT_ACCUMULATION = 4  # Effective batch = BATCH_SIZE * GRADIENT_ACCUMULATION

# Gemma 3 does NOT officially support system role in its chat template.
# System instructions are embedded in the first user message instead.
SYSTEM_PROMPT = (
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0bb5\u0bb4\u0bbf (VAZHI), "
    "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bc1 \u0bae\u0b95\u0bcd\u0b95\u0bb3\u0bc1\u0b95\u0bcd\u0b95\u0bbe\u0ba9 "
    "AI \u0b89\u0ba4\u0bb5\u0bbf\u0baf\u0bbe\u0bb3\u0bb0\u0bcd. "
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0ba4\u0bae\u0bbf\u0bb4\u0bbf\u0bb2\u0bcd \u0baa\u0ba4\u0bbf\u0bb2\u0bb3\u0bbf\u0baa\u0bcd\u0baa\u0bc0\u0bb0\u0bcd\u0b95\u0bb3\u0bcd."
)

# GPU auto-detection
assert torch.cuda.is_available(), "GPU required! Runtime > Change runtime type > GPU"
gpu_name = torch.cuda.get_device_name(0).lower()
_props = torch.cuda.get_device_properties(0)
VRAM_GB = getattr(_props, 'total_memory', getattr(_props, 'total_mem', 0)) / 1e9
IS_HIGH_END_GPU = any(x in gpu_name for x in ["a100", "l4", "h100", "a10"])
USE_BF16 = IS_HIGH_END_GPU
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
n_gpus = torch.cuda.device_count()
effective_batch = BATCH_SIZE * n_gpus * GRADIENT_ACCUMULATION

print(f"\u2705 SFT v7.0 Configuration (Gemma 3):")
print(f"   Environment: {ENV_NAME}")
print(f"   Base model:  {BASE_MODEL} (native Tamil, no DAPT needed)")
print(f"   Dataset:     {SFT_DATASET} (v7.0 rebalanced)")
print(f"   Output:      {OUTPUT_MODEL}")
print(f"   LR:          {LEARNING_RATE}")
print(f"   LoRA:        r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGETS}")
print(f"   Batch:       {BATCH_SIZE} x {GRADIENT_ACCUMULATION} accum = {effective_batch} effective")
print(f"   GPU:         {torch.cuda.get_device_name(0)} ({VRAM_GB:.0f} GB)")
print(f"   Precision:   {'bf16' if USE_BF16 else 'fp16'}")

✅ SFT v7.0 Configuration (Gemma 3):
   Environment: Colab
   Base model:  google/gemma-3-1b-it (native Tamil, no DAPT needed)
   Dataset:     CryptoYogi/vazhi-tamil-sft-v7_0 (v7.0 rebalanced)
   Output:      CryptoYogi/vazhi-v7_0
   LR:          1e-05
   LoRA:        r=8, alpha=16, targets=['q_proj', 'v_proj']
   Batch:       4 x 4 accum = 16 effective
   GPU:         NVIDIA L4 (24 GB)
   Precision:   bf16


In [3]:
# Cell 3 — HuggingFace Login
from huggingface_hub import notebook_login
notebook_login()
print("\u2705 Logged in to HuggingFace")

✅ Logged in to HuggingFace


In [4]:
# Cell 4 — Load Tokenizer + Helper Functions

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\u2705 Tokenizer: {len(tokenizer)} tokens")
print(f"   pad_token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"   eos_token: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")

# Verify chat template works
test_msgs = [{"role": "user", "content": "test"}]
test_formatted = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
print(f"   Chat template test: {repr(test_formatted[:80])}")

# Check if system role is supported in the template
SYSTEM_ROLE_SUPPORTED = True
try:
    test_sys = [
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "test"},
    ]
    sys_formatted = tokenizer.apply_chat_template(test_sys, tokenize=False, add_generation_prompt=True)
    print(f"   System role: supported")
    print(f"   System test: {repr(sys_formatted[:120])}")
except Exception as e:
    SYSTEM_ROLE_SUPPORTED = False
    print(f"   System role: NOT supported ({e})")
    print(f"   Will embed system prompt in first user message")


def build_messages(user_text, system_text=None):
    """Build message list for Gemma 3 chat template."""
    msgs = []
    if system_text and SYSTEM_ROLE_SUPPORTED:
        msgs.append({"role": "system", "content": system_text})
        msgs.append({"role": "user", "content": user_text})
    elif system_text:
        # Embed system prompt in first user message
        msgs.append({"role": "user", "content": f"{system_text}\n\n{user_text}"})
    else:
        msgs.append({"role": "user", "content": user_text})
    return msgs


def build_chat_prompt(user_text):
    """Build a formatted prompt string for generation."""
    msgs = build_messages(user_text, SYSTEM_PROMPT)
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True,
    )


def tamil_char_pct(text):
    if not text:
        return 0.0
    total = sum(1 for c in text if not c.isspace() and not c.isdigit())
    if total == 0:
        return 0.0
    tamil = sum(1 for c in text if '\u0B80' <= c <= '\u0BFF')
    return 100.0 * tamil / total


def tamil_word_score(text):
    """Score based on Tamil word validation (not just char %)."""
    words = text.split()
    if not words:
        return 0.0, 0, 0
    tamil_words = 0
    for w in words:
        clean = re.sub(r'[\d\W]', '', w)
        if not clean:
            continue
        tamil_chars = sum(1 for c in clean if '\u0B80' <= c <= '\u0BFF')
        if tamil_chars / len(clean) > 0.5:
            tamil_words += 1
    return 100.0 * tamil_words / len(words), tamil_words, len(words)


def compute_repeat_ratio(text, n=3):
    """Detect repetitive output via n-gram ratio."""
    words = text.split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    if not ngrams:
        return 0.0
    return 1.0 - len(set(ngrams)) / len(ngrams)


def extract_response(full_text):
    """Extract model response from Gemma formatted output."""
    # Gemma format: <start_of_turn>model\n{response}<end_of_turn>
    if "<start_of_turn>model" in full_text:
        # Get the last model turn
        parts = full_text.split("<start_of_turn>model")
        resp = parts[-1]
        if "<end_of_turn>" in resp:
            resp = resp.split("<end_of_turn>")[0]
        resp = resp.strip()
        if resp.startswith("\n"):
            resp = resp[1:]
        return resp.strip()
    return full_text.strip()


def generate_response(model, prompt_text, max_new_tokens=200):
    """Generate a response using Gemma 3 chat format."""
    full_prompt = build_chat_prompt(prompt_text)
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only new tokens
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=False)

    # Clean up end-of-turn markers
    if "<end_of_turn>" in response:
        response = response.split("<end_of_turn>")[0]
    # Remove any remaining special tokens
    response = response.replace("<eos>", "").replace("<bos>", "").strip()
    return response


print("\u2705 Helpers ready")

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

✅ Tokenizer: 262145 tokens
   pad_token: <pad> (id=0)
   eos_token: <eos> (id=1)
   Chat template test: '<bos><start_of_turn>user\ntest<end_of_turn>\n<start_of_turn>model\n'
   System role: supported
   System test: '<bos><start_of_turn>user\nYou are helpful.\n\ntest<end_of_turn>\n<start_of_turn>model\n'
✅ Helpers ready


In [5]:
# Cell 5 — Pre-SFT Baseline on Vanilla Gemma 3 1B-it
#
# Record vanilla model outputs BEFORE SFT for comparison.
# Gemma 3 already produces real Tamil — SFT should improve
# domain accuracy and VAZHI identity without degrading Tamil quality.

print("\U0001f4ca Pre-SFT Baseline: Vanilla Gemma 3 1B-it")
print("=" * 60)

BASELINE_PROMPTS = [
    ("\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "greeting"),
    ("\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "identity"),
    ("\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf", "thanks"),
    ("\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bbf\u0ba9\u0bcd \u0ba4\u0bb2\u0bc8\u0ba8\u0b95\u0bb0\u0bae\u0bcd \u0b8e\u0ba4\u0bc1?", "factual"),
    ("\u0bb0\u0bc7\u0bb7\u0ba9\u0bcd \u0b95\u0bbe\u0bb0\u0bcd\u0b9f\u0bc1 \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0ba4\u0b95\u0bb5\u0bb2\u0bcd \u0ba4\u0bc7\u0bb5\u0bc8", "govt"),
    ("\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "govt"),
    ("\u0ba8\u0bc0\u0bb0\u0bbf\u0bb4\u0bbf\u0bb5\u0bc1 \u0ba8\u0bcb\u0baf\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "health"),
    ("\u0b92\u0bb0\u0bc1 \u0ba4\u0bc6\u0bb0\u0bbf\u0baf\u0bbe\u0ba4 \u0b8e\u0ba3\u0bcd\u0ba3\u0bbf\u0bb2\u0bcd \u0b87\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bc1 \u0bae\u0bc6\u0b9a\u0bc7\u0b9c\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0ba4\u0bc1", "safety"),
    ("\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bc1\u0bb1\u0bb3\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "culture"),
    ("\u0b95\u0bbe\u0baf\u0bcd\u0b9a\u0bcd\u0b9a\u0bb2\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf \u0bb5\u0bc7\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd?", "health"),
]

print(f"\n\U0001f4e5 Loading {BASE_MODEL} for baseline...")
baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=MODEL_DTYPE, device_map={"":0},
)
baseline_model.eval()
baseline_model.config.use_cache = True

pre_sft_results = []
for prompt_text, category in BASELINE_PROMPTS:
    resp = generate_response(baseline_model, prompt_text)
    t_pct = tamil_char_pct(resp)
    tw_pct, tw_count, tw_total = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    pre_sft_results.append({
        'prompt': prompt_text, 'category': category,
        'response': resp[:300], 'tamil_char_pct': t_pct,
        'tamil_word_pct': tw_pct, 'repeat_ratio': rep,
    })
    print(f"\n[{category}] Char: {t_pct:.0f}%, Word: {tw_pct:.0f}%, Rep: {rep:.2f}")
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")

avg_pre_char = np.mean([r['tamil_char_pct'] for r in pre_sft_results])
avg_pre_word = np.mean([r['tamil_word_pct'] for r in pre_sft_results])
avg_pre_rep = np.mean([r['repeat_ratio'] for r in pre_sft_results])
print(f"\n\U0001f4ca Vanilla baseline: char {avg_pre_char:.0f}%, word {avg_pre_word:.0f}%, rep {avg_pre_rep:.2f}")

del baseline_model
gc.collect(); torch.cuda.empty_cache()
print("\U0001f5d1\ufe0f Baseline model freed")

📊 Pre-SFT Baseline: Vanilla Gemma 3 1B-it

📥 Loading google/gemma-3-1b-it for baseline...


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]


[greeting] Char: 96%, Word: 93%, Rep: 0.00
  Q: வணக்கம்
  A: வணக்கம்! நான் உங்களுக்கு எப்படி உதவ முடியும்? என்னிடம் ஏதாவது கேள்விகள் இருக்கிறதா அல்லது உதவி வேண்டுமா? 😊

[identity] Char: 97%, Word: 100%, Rep: 0.00
  Q: நீங்கள் யார்?
  A: நான் ஒரு பெரிய மொழி மாதிரி, கூகிளால் பயிற்சி அளிக்கப்பட்டது. நான் எப்படிப் பதில் சொல்வது என்று எனக்குத் தெரியவில்லை.

[thanks] Char: 94%, Word: 93%, Rep: 0.00
  Q: நன்றி
  A: வணக்கம்! நான் VAZI, உங்கள் மொழி சார்ந்த உதவிக்கு தயாராக இருக்கிறேன். உங்களுக்கு என்ன தேவை? எப்படி உதவலாம் என்று சொல்லுங்கள். 😊 

உங்களுக்கு ஏதேனும் கேள்விகள் இருந்தாலோ அல்லது ஏதாவது செய்ய வேண்டியிருந்

[factual] Char: 97%, Word: 100%, Rep: 0.00
  Q: தமிழ்நாட்டின் தலைநகரம் எது?
  A: தமிழ்நாட்டின் தற்போதைய தலைநகரம் சென்னையில் உள்ளது. ஆனால், முன்பு சென்னைதான் தான் முழு நகரம்! தற்போது, புதுத்தொடவேயில் இருந்து ஆக்சபாவூர் வரை உள்ள நகரங்கள் எல்லாம் தமிழ்நாட்டின் தலைநகரங்களாக உள்ளன. 

உ

[govt] Char: 80%, Word: 92%, Rep: 0.00
  Q: ரேஷன் கார்டு பற்றி தகவல் தேவை
  A: சாரி, ரேஷன் கார்டை பற்ற

In [6]:
# Cell 6 — Load & Format Dataset v7.0 (instruction/output → Gemma format)
#
# v7.0 dataset has raw instruction/output columns (NOT ChatML).
# We apply Gemma 3's chat template here at training time.
# This is cleaner than v5.3's ChatML→Gemma conversion hack.

print(f"\U0001f4da Loading SFT dataset from {SFT_DATASET}...")

try:
    sft_ds = load_dataset(SFT_DATASET)
    if "train" in sft_ds and "validation" in sft_ds:
        raw_train = sft_ds["train"]
        raw_eval = sft_ds["validation"]
    elif "train" in sft_ds:
        raw_train = sft_ds["train"]
        raw_eval = None
    else:
        raise KeyError("No train split found")
except (KeyError, ValueError, Exception) as e:
    print(f"   HF load failed ({e}), trying JSON files...")
    raw_train = load_dataset("json", data_files={
        "train": f"hf://datasets/{SFT_DATASET}/vazhi-tamil-sft-v7_0-train.json"
    })["train"]
    raw_eval = load_dataset("json", data_files={
        "eval": f"hf://datasets/{SFT_DATASET}/vazhi-tamil-sft-v7_0-eval.json"
    })["eval"]

print(f"   Raw train: {len(raw_train)}")
print(f"   Raw eval:  {len(raw_eval) if raw_eval else 'None'}")
print(f"   Columns: {raw_train.column_names}")

# === Apply Gemma 3 chat template ===
def format_for_gemma(sample):
    """Convert instruction/output pair to Gemma chat format."""
    instruction = sample["instruction"]
    output = sample["output"]

    if not instruction or not output:
        return None

    # Build messages with system prompt embedded in user turn
    messages = build_messages(instruction, SYSTEM_PROMPT)

    # Add model response (Gemma uses "model" not "assistant")
    messages.append({"role": "model", "content": output})

    # Format using Gemma tokenizer's chat template
    try:
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )
    except Exception:
        # Fallback: try with "assistant" role if "model" fails
        messages[-1]["role"] = "assistant"
        try:
            formatted = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False,
            )
        except Exception as e2:
            print(f"   Template error: {e2}")
            return None

    return {
        "text": formatted,
        "bucket": sample.get("bucket", "unknown"),
    }


def convert_dataset(raw_ds, split_name):
    """Convert all samples in a dataset split."""
    converted = []
    failed = 0
    for i in range(len(raw_ds)):
        result = format_for_gemma(raw_ds[i])
        if result:
            converted.append(result)
        else:
            failed += 1
    print(f"   {split_name}: {len(raw_ds)} → {len(converted)} converted, {failed} failed")
    return Dataset.from_list(converted)


print("\n\U0001f500 Applying Gemma 3 chat template...")
train_ds = convert_dataset(raw_train, "train")
if raw_eval:
    eval_ds = convert_dataset(raw_eval, "eval")
else:
    # If no eval split, create from train (10%)
    full_ds = convert_dataset(raw_train, "full")
    split = full_ds.train_test_split(test_size=0.1, seed=RANDOM_SEED)
    train_ds = split["train"]
    eval_ds = split["test"]
    print(f"   Split: {len(train_ds)} train / {len(eval_ds)} eval")

print(f"\n\u2705 Formatted dataset:")
print(f"   Train: {len(train_ds)} samples")
print(f"   Eval:  {len(eval_ds)} samples")

# Spot-check: show first converted sample
print(f"\n\U0001f50d Sample (first 500 chars):")
print(train_ds[0]["text"][:500])

# Verify Gemma format tokens are present
sample_text = train_ds[0]["text"]
has_start = "<start_of_turn>" in sample_text
has_end = "<end_of_turn>" in sample_text
has_model = "<start_of_turn>model" in sample_text
has_user = "<start_of_turn>user" in sample_text
print(f"\n   Format check: start_of_turn={has_start}, end_of_turn={has_end}, model={has_model}, user={has_user}")
assert has_start and has_end and has_model and has_user, "Gemma format conversion failed!"
print("\u2705 Gemma format verified")

# Composition stats
if 'bucket' in train_ds.column_names:
    bucket_dist = Counter(train_ds['bucket'])
    total_count = sum(bucket_dist.values())
    print(f"\n\U0001f4ca Composition (train):")
    for bucket, count in sorted(bucket_dist.items(), key=lambda x: -x[1]):
        print(f"   {bucket}: {count} ({100*count/total_count:.1f}%)")

📚 Loading SFT dataset from CryptoYogi/vazhi-tamil-sft-v7_0...


README.md: 0.00B [00:00, ?B/s]

vazhi-tamil-sft-v7_0-train.json: 0.00B [00:00, ?B/s]

vazhi-tamil-sft-v7_0-eval.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3754 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/418 [00:00<?, ? examples/s]

   Raw train: 3754
   Raw eval:  None
   Columns: ['instruction', 'output', 'bucket', 'source', 'category']

🔀 Applying Gemma 3 chat template...
   train: 3754 → 3754 converted, 0 failed
   full: 3754 → 3754 converted, 0 failed
   Split: 3378 train / 376 eval

✅ Formatted dataset:
   Train: 3378 samples
   Eval:  376 samples

🔍 Sample (first 500 chars):
<bos><start_of_turn>user
நீங்கள் வழி (VAZHI), தமிழ்நாட்டு மக்களுக்கான AI உதவியாளர். நீங்கள் தமிழில் பதிலளிப்பீர்கள்.

கடை தொடங்க என்ன அனுமதி வேண்டும்?<end_of_turn>
<start_of_turn>model
1. GST பதிவு (₹40 லட்சத்திற்கு மேல் விற்பனை) 2. வர்த்தக உரிமம்: நகராட்சி/கிராம பஞ்சாயத்து 3. கடை & நிறுவனச் சட்டப் பதிவு: tnlabour.tn.gov.in 4. FSSAI உரிமம் (உணவு வணிகம்): fssai.gov.in 5. தீ பாதுகாப்புச் சான்றிதழ் (பெரிய கடைகள்)<end_of_turn>


   Format check: start_of_turn=True, end_of_turn=True, model=True, user=True
✅ Gemma format verified

📊 Composition (train):
   vazhi_packs: 2400 (71.0%)
   sadhguru_qa: 318 (9.4%)
   conversational: 217 (6.4%)
   t

In [7]:
# Cell 7 — Load Model + LoRA Setup
#
# CONSERVATIVE LoRA to avoid Navarasa-style catastrophic forgetting.
# Navarasa 2.0 SFT on Gemma destroyed instruction-following.
# We use r=8 on q_proj+v_proj only — minimal intervention.

print(f"\U0001f4e5 Loading {BASE_MODEL}...")

# NO device_map for training — use .to("cuda:0") instead
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=MODEL_DTYPE,
)
model = model.to("cuda:0")

if tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable()

has_device_map = hasattr(model, "hf_device_map")
print(f"   hf_device_map present: {has_device_map} (must be False)")

mem_gb = torch.cuda.memory_allocated(0) / 1024**3
print(f"\u2705 Model loaded: {model.num_parameters():,} params | GPU: {mem_gb:.1f} GB")

# Verify LoRA target modules exist in the model
print(f"\n\U0001f50d Checking LoRA target modules...")
found_modules = set()
for name, _ in model.named_modules():
    for target in LORA_TARGETS:
        if target in name:
            found_modules.add(target)
for target in LORA_TARGETS:
    status = "\u2705" if target in found_modules else "\u274c MISSING"
    print(f"   {target}: {status}")
assert len(found_modules) == len(LORA_TARGETS), f"Missing LoRA targets: {set(LORA_TARGETS) - found_modules}"

# Apply LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

mem_gb = torch.cuda.memory_allocated(0) / 1024**3
print(f"\u2705 LoRA applied | GPU: {mem_gb:.1f} GB")

📥 Loading google/gemma-3-1b-it...
   hf_device_map present: False (must be False)
✅ Model loaded: 999,885,952 params | GPU: 1.9 GB

🔍 Checking LoRA target modules...
   q_proj: ✅
   v_proj: ✅
trainable params: 745,472 || all params: 1,000,631,424 || trainable%: 0.0745
✅ LoRA applied | GPU: 1.9 GB


In [8]:
# Cell 8 — Dataset Preflight: Token Length Validation

print(f"\U0001f4ca Token length distribution (train):")
token_lengths = []
for idx in range(len(train_ds)):
    tokens = tokenizer.encode(train_ds[idx]["text"], add_special_tokens=False)
    token_lengths.append(len(tokens))

token_lengths = np.array(token_lengths)
truncated = (token_lengths > MAX_LENGTH).sum()

print(f"   Mean: {token_lengths.mean():.0f}, Max: {token_lengths.max()}, P95: {np.percentile(token_lengths, 95):.0f}")
print(f"   Truncated (>{MAX_LENGTH}): {truncated} ({100*truncated/len(train_ds):.1f}%)")

# Gemma 3 uses 262K vocab — tokenization may differ from Qwen3 (151K)
# Compare average tokens per sample
avg_tokens = token_lengths.mean()
print(f"   Avg tokens/sample: {avg_tokens:.0f}")
print(f"   Total tokens: {token_lengths.sum():,}")

if truncated > len(train_ds) * 0.2:
    print(f"   \u26a0\ufe0f >20% truncated — consider increasing MAX_LENGTH or filtering long samples")
elif truncated > len(train_ds) * 0.1:
    print(f"   \u26a0\ufe0f >10% truncated — acceptable but some Sadhguru articles will be cut")
else:
    print(f"   \u2705 Token lengths OK")

📊 Token length distribution (train):
   Mean: 167, Max: 1200, P95: 576
   Truncated (>2048): 0 (0.0%)
   Avg tokens/sample: 167
   Total tokens: 564,559
   ✅ Token lengths OK


In [9]:
# Cell 9 — Training Setup

OUTPUT_DIR = f"{WORK_DIR}/sft-v7_0"

steps_per_epoch = len(train_ds) // effective_batch
total_steps = steps_per_epoch * NUM_EPOCHS
log_steps = max(total_steps // 30, 5)
eval_steps = max(steps_per_epoch // 3, 10)
save_steps = max(steps_per_epoch, 20)

print(f"\U0001f4ca Training Plan:")
print(f"   Train samples:    {len(train_ds)}")
print(f"   Eval samples:     {len(eval_ds)}")
print(f"   Effective batch:  {effective_batch}")
print(f"   Steps/epoch:      ~{steps_per_epoch}")
print(f"   Total steps:      ~{total_steps}")
print(f"   Eval every:       {eval_steps} steps")
print(f"   LoRA:             r={LORA_R}, targets={LORA_TARGETS}")


class LossLoggingCallback(TrainerCallback):
    def __init__(self):
        self.losses = []
        self.eval_losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if "loss" in logs:
                step = state.global_step
                loss = logs["loss"]
                lr = logs.get("learning_rate", 0)
                self.losses.append((step, loss))
                print(f"  Step {step:4d}/{total_steps} | Loss: {loss:.4f} | LR: {lr:.2e}")
            if "eval_loss" in logs:
                self.eval_losses.append((state.global_step, logs["eval_loss"]))
                print(f"  \U0001f4ca Eval Loss: {logs['eval_loss']:.4f}")


class MidTrainingGenCheck(TrainerCallback):
    """Generate Tamil responses mid-training to catch catastrophic forgetting early."""

    SANITY_PROMPTS = [
        {"prompt": "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "label": "greeting"},
        {"prompt": "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "label": "identity"},
        {"prompt": "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bbf\u0ba9\u0bcd \u0ba4\u0bb2\u0bc8\u0ba8\u0b95\u0bb0\u0bae\u0bcd \u0b8e\u0ba4\u0bc1?", "label": "factual"},
    ]

    def __init__(self, model_ref):
        self.model_ref = model_ref
        self.check_interval = max(steps_per_epoch // 2, 20)

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.check_interval != 0 or state.global_step == 0:
            return
        print(f"\n  \U0001f50d Mid-training gen check (step {state.global_step}):")
        self.model_ref.eval()
        self.model_ref.config.use_cache = True
        if hasattr(self.model_ref, 'gradient_checkpointing_disable'):
            self.model_ref.gradient_checkpointing_disable()

        for item in self.SANITY_PROMPTS:
            try:
                resp = generate_response(self.model_ref, item['prompt'], max_new_tokens=80)
                tw_pct, _, _ = tamil_word_score(resp)
                rep = compute_repeat_ratio(resp)
                print(f"    [{item['label']}] Word: {tw_pct:.0f}%, Rep: {rep:.2f} | {resp[:100]}")
            except Exception as e:
                print(f"    [{item['label']}] ERROR: {e}")

        self.model_ref.train()
        self.model_ref.config.use_cache = False
        if hasattr(self.model_ref, 'gradient_checkpointing_enable'):
            self.model_ref.gradient_checkpointing_enable()


loss_cb = LossLoggingCallback()
gen_cb = MidTrainingGenCheck(model)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=log_steps,
    save_steps=save_steps,
    eval_steps=eval_steps,
    eval_strategy="steps",
    save_total_limit=3,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=1.0,
    optim="adamw_torch",
    report_to="none",
    seed=RANDOM_SEED,
    load_best_model_at_end=False,
    dataloader_pin_memory=True,
    max_length=MAX_LENGTH,
    packing=False,
    push_to_hub=True,
    hub_model_id=ADAPTER_REPO,
    hub_strategy="every_save",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
    processing_class=tokenizer,
    callbacks=[loss_cb, gen_cb],
)

print(f"\u2705 SFTTrainer ready")
print(f"   Base: {BASE_MODEL} (Gemma 3 1B-it, native Tamil)")
print(f"   LR: {LEARNING_RATE}, Epochs: {NUM_EPOCHS}")
print(f"   LoRA: r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGETS}")

📊 Training Plan:
   Train samples:    3378
   Eval samples:     376
   Effective batch:  16
   Steps/epoch:      ~211
   Total steps:      ~211
   Eval every:       70 steps
   LoRA:             r=8, targets=['q_proj', 'v_proj']


Adding EOS to train dataset:   0%|          | 0/3378 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3378 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3378 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/376 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/376 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/376 [00:00<?, ? examples/s]

✅ SFTTrainer ready
   Base: google/gemma-3-1b-it (Gemma 3 1B-it, native Tamil)
   LR: 1e-05, Epochs: 1
   LoRA: r=8, alpha=16, targets=['q_proj', 'v_proj']


In [10]:
# Cell 10 — Run Training

print(f"\U0001f680 Starting SFT v7.0 training (Gemma 3)...")
print(f"   ~{total_steps} steps, {NUM_EPOCHS} epoch")
print(f"   Base: {BASE_MODEL}")
print(f"   LR: {LEARNING_RATE}, Dataset: {len(train_ds)} train")
print()

train_result = trainer.train()

print("\n\u2705 Training complete!")
metrics = train_result.metrics
for k, v in metrics.items():
    print(f"   {k}: {v}")

print("\n\U0001f4ca Final eval...")
eval_metrics = trainer.evaluate()
for k, v in eval_metrics.items():
    print(f"   {k}: {v}")

if loss_cb.losses:
    s = loss_cb.losses[0][1]
    e = loss_cb.losses[-1][1]
    print(f"\n\U0001f4c8 Loss: {s:.4f} \u2192 {e:.4f} ({100*(s-e)/s:.1f}% drop)")

trainer.save_model()
trainer.push_to_hub()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


🚀 Starting SFT v7.0 training (Gemma 3)...
   ~211 steps, 1 epoch
   Base: google/gemma-3-1b-it
   LR: 1e-05, Dataset: 3378 train



Step,Training Loss,Validation Loss
70,4.938600,5.066318
140,4.647900,4.801804
210,4.732200,4.753093


  Step    7/211 | Loss: 5.3522 | LR: 2.73e-06
  Step   14/211 | Loss: 5.2669 | LR: 5.91e-06
  Step   21/211 | Loss: 5.5322 | LR: 9.09e-06
  Step   28/211 | Loss: 5.3355 | LR: 9.98e-06
  Step   35/211 | Loss: 5.1915 | LR: 9.90e-06
  Step   42/211 | Loss: 5.2243 | LR: 9.76e-06
  Step   49/211 | Loss: 5.1027 | LR: 9.55e-06
  Step   56/211 | Loss: 5.0369 | LR: 9.27e-06
  Step   63/211 | Loss: 5.1718 | LR: 8.95e-06
  Step   70/211 | Loss: 4.9386 | LR: 8.56e-06
  📊 Eval Loss: 5.0663
  Step   77/211 | Loss: 5.1139 | LR: 8.14e-06
  Step   84/211 | Loss: 4.9361 | LR: 7.67e-06
  Step   91/211 | Loss: 5.1318 | LR: 7.16e-06
  Step   98/211 | Loss: 4.9415 | LR: 6.62e-06

  🔍 Mid-training gen check (step 105):
    [greeting] Word: 46%, Rep: 0.00 | வணக்கம்! நான் உங்களுக்கு எப்படி உதவ முடியும்? என்ன கேட்க விரும்புகிறீர்களோ, தயங்காமல் கேளுங்கள். 

(
    [identity] Word: 100%, Rep: 0.00 | நான் கூகிள் மூலம் பயிற்சி பெற்ற ஒரு பெரிய மொழி மாதிரி, நான் ஒரு உதவி மையமாக செயல்படுகிறேன். நான் எந்
    [factual] W

  📊 Eval Loss: 4.7532
   eval_loss: 4.753226280212402
   eval_runtime: 20.5095
   eval_samples_per_second: 18.333
   eval_steps_per_second: 4.583

📈 Loss: 5.3522 → 4.7322 (11.6% drop)


...

  ...adapter_model.safetensors: 100%|##########| 3.00MB / 3.00MB            

  .../sft-v7_0/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...t/sft-v7_0/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/CryptoYogi/vazhi-v7_0-lora/commit/e769d1a674b7c78ad17fa8920766fd081e575b86', commit_message='End of training', commit_description='', oid='e769d1a674b7c78ad17fa8920766fd081e575b86', pr_url=None, repo_url=RepoUrl('https://huggingface.co/CryptoYogi/vazhi-v7_0-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='CryptoYogi/vazhi-v7_0-lora'), pr_revision=None, pr_num=None)

In [ ]:
# Cell 10r — Resume Training (if Colab disconnected)
# Uncomment and run ONLY if Cell 10 was interrupted

# print("Resuming training from checkpoint...")
# train_result = trainer.train(resume_from_checkpoint=True)
# trainer.save_model()
# trainer.push_to_hub()

print("Cell 10r: Resume cell (commented out). Uncomment only if needed.")

In [11]:
# Cell 11 — Save Adapter + A/B Test + Merge
#
# Test BOTH adapter inference and merged inference.
# If adapter works but merged doesn't, the merge is the problem.
# CRITICAL: Merge in fp16, NEVER into 4-bit (lesson from v3.6).

ADAPTER_PATH = f"{WORK_DIR}/vazhi-sft-v7_0-lora"

print("\U0001f4be Saving LoRA adapter...")
trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"\u2705 Adapter saved to {ADAPTER_PATH}")

# Upload adapter backup
api = HfApi()
api.create_repo(ADAPTER_REPO, exist_ok=True)
print(f"\U0001f4e4 Uploading adapter to {ADAPTER_REPO}...")
api.upload_folder(
    folder_path=ADAPTER_PATH,
    repo_id=ADAPTER_REPO,
    commit_message=f"SFT v7.0 adapter: Gemma 3 1B-it, {len(train_ds)} samples, r={LORA_R}, lr={LEARNING_RATE}",
)
print(f"\u2705 Adapter uploaded")

# Free training model
del model, trainer
gc.collect(); torch.cuda.empty_cache()
print("\U0001f5d1\ufe0f Training model freed")

AB_PROMPTS = [
    "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd",
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?",
    "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bbf\u0ba9\u0bcd \u0ba4\u0bb2\u0bc8\u0ba8\u0b95\u0bb0\u0bae\u0bcd \u0b8e\u0ba4\u0bc1?",
    "\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd",
    "\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf",
]

# --- Test A: Adapter inference ---
print("\n" + "=" * 60)
print("\U0001f1e6 TEST A: Adapter Inference (no merge)")
print("=" * 60)

base_a = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"":0},
)
adapter_model = PeftModel.from_pretrained(base_a, ADAPTER_PATH)
adapter_model.eval()
adapter_model.config.use_cache = True

adapter_results = []
for prompt_text in AB_PROMPTS:
    resp = generate_response(adapter_model, prompt_text)
    tw_pct, _, _ = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    adapter_results.append({"resp": resp, "word": tw_pct, "rep": rep})
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")
    print(f"  (Word:{tw_pct:.0f}% Rep:{rep:.2f})")
    print()

del adapter_model, base_a
gc.collect(); torch.cuda.empty_cache()

# --- Test B: Merged model ---
print("\n" + "=" * 60)
print("\U0001f1e7 TEST B: Merged Model (fp16)")
print("=" * 60)

base_b = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"":0},
)
peft_b = PeftModel.from_pretrained(base_b, ADAPTER_PATH)
peft_b.gradient_checkpointing_disable()
peft_b.config.use_cache = True
peft_b.eval()

print("\U0001f500 Merging LoRA in fp16...")
merged_model = peft_b.merge_and_unload()

merged_results = []
for prompt_text in AB_PROMPTS:
    resp = generate_response(merged_model, prompt_text)
    tw_pct, _, _ = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    merged_results.append({"resp": resp, "word": tw_pct, "rep": rep})
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")
    print(f"  (Word:{tw_pct:.0f}% Rep:{rep:.2f})")
    print()

# A/B comparison
print("\n" + "=" * 60)
print("A/B COMPARISON")
print("=" * 60)
avg_a_word = np.mean([r['word'] for r in adapter_results])
avg_b_word = np.mean([r['word'] for r in merged_results])
print(f"   Adapter avg word: {avg_a_word:.0f}%")
print(f"   Merged avg word:  {avg_b_word:.0f}%")
if abs(avg_a_word - avg_b_word) > 15:
    print(f"   \u26a0\ufe0f MERGE CORRUPTION DETECTED \u2014 use adapter for deployment")
else:
    print(f"   \u2705 Merge OK \u2014 adapter and merged consistent")

💾 Saving LoRA adapter...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ft-v7_0/training_args.bin: 100%|##########| 6.29kB / 6.29kB            

  ...adapter_model.safetensors: 100%|##########| 3.00MB / 3.00MB            

...

  ..._0-lora/training_args.bin: 100%|##########| 6.29kB / 6.29kB            

  ...adapter_model.safetensors: 100%|##########| 3.00MB / 3.00MB            

  ...v7_0-lora/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...-v7_0-lora/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

✅ Adapter uploaded
🗑️ Training model freed

🇦 TEST A: Adapter Inference (no merge)
  Q: வணக்கம்
  A: வணக்கம்! நான் VAZHI, உங்கள் தமிழ் மொழி பேசும் AI உதவியாளராக இருக்கிறேன். உங்களுக்கு என்ன உதவி வேண்டும்? 

எப்படி உதவலாம் என்று சொல்லுங்கள். 😊
  (Word:84% Rep:0.00)

  Q: நீங்கள் யார்?
  A: நான் ஒரு பெரிய மொழி மாதிரி, கூகிளால் பயிற்சி அளிக்கப்பட்டது. நான் எப்படி உதவ முடியும் என்று எனக்குத் தெரியப்படுத்துங்கள்!
  (Word:100% Rep:0.00)

  Q: தமிழ்நாட்டின் தலைநகரம் எது?
  A: தமிழ்நாட்டின் தற்போதைய தலைநகரம் கோயம்புத்தூர். 

இது ஒரு சிறிய நகரம், ஆனால் இது முக்கிய நகரமாகும்.
  (Word:100% Rep:0.00)

  Q: முதியோர் ஓய்வூதியம் பற்றி சொல்லுங்கள்
  A: நிச்சயமாக! முதியோருக்கான ஓய்வூதியங்கள் பற்றிய தகவல்கள் இதோ:

**முதியோருவதற்கான ஓய்வூட்டல்கள் என்றால் என்ன?**

முதியோர்கள் ஓய்வூட்டி என்பது, வயதானவர்கள் முதுமை காரணமாக கிடைக்கும் வருமானத்தை ஈடுசெய்யும்
  (Word:95% Rep:0.00)

  Q: நன்றி
  A: சாரி, எனக்குப் புரியல. தயவுசெய்து வேறொரு வழியில் சொல்ல முடியுமா?
  (Word:100% Rep:0.00)


🇧 TEST B: Merged Model (f

In [12]:
# Cell 12 — Full Eval: 16 Conversational Prompts
#
# Tamil WORD validation + human-readable output for quality assessment.
# Key question: Does SFT improve VAZHI identity/domain knowledge
# WITHOUT destroying Gemma 3's native Tamil capability?

merged_model.eval()
merged_model.config.use_cache = True

test_prompts = [
    {"prompt": "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "check": "greeting", "cat": "greeting"},
    {"prompt": "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "check": "identity", "cat": "greeting"},
    {"prompt": "\u0b8e\u0ba9\u0b95\u0bcd\u0b95\u0bc1 \u0b89\u0ba4\u0bb5\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "general", "cat": "help"},
    {"prompt": "\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf", "check": "general", "cat": "help"},
    {"prompt": "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bbf\u0ba9\u0bcd \u0ba4\u0bb2\u0bc8\u0ba8\u0b95\u0bb0\u0bae\u0bcd \u0b8e\u0ba4\u0bc1?", "check": "factual", "cat": "factual"},
    {"prompt": "\u0b92\u0bb0\u0bc1 \u0ba4\u0bc6\u0bb0\u0bbf\u0baf\u0bbe\u0ba4 \u0b8e\u0ba3\u0bcd\u0ba3\u0bbf\u0bb2\u0bcd \u0b87\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bc1 \u0bae\u0bc6\u0b9a\u0bc7\u0b9c\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0ba4\u0bc1. \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0bb5\u0ba4\u0bc1?", "check": "safety", "cat": "safety"},
    {"prompt": "\u0bb5\u0bc0\u0b9f\u0bcd\u0b9f\u0bbf\u0bb2\u0bcd \u0ba4\u0bc0 \u0bb5\u0bbf\u0baa\u0ba4\u0bcd\u0ba4\u0bc1 \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf \u0bb5\u0bc7\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd?", "check": "safety", "cat": "safety"},
    {"prompt": "\u0ba8\u0bbe\u0bb3\u0bc8 \u0baa\u0b99\u0bcd\u0b95\u0bc1 \u0b9a\u0ba8\u0bcd\u0ba4\u0bc8 \u0b8f\u0bb1\u0bc1\u0bae\u0bbe?", "check": "refusal", "cat": "refusal"},
    {"prompt": "\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "govt"},
    {"prompt": "\u0bb0\u0bc7\u0bb7\u0ba9\u0bcd \u0b95\u0bbe\u0bb0\u0bcd\u0b9f\u0bc1 \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0ba4\u0b95\u0bb5\u0bb2\u0bcd \u0ba4\u0bc7\u0bb5\u0bc8", "check": "domain", "cat": "govt"},
    {"prompt": "\u0ba8\u0bc0\u0bb0\u0bbf\u0bb4\u0bbf\u0bb5\u0bc1 \u0ba8\u0bcb\u0baf\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "health"},
    {"prompt": "\u0b95\u0bbe\u0baf\u0bcd\u0b9a\u0bcd\u0b9a\u0bb2\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf \u0bb5\u0bc7\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd?", "check": "domain", "cat": "health"},
    {"prompt": "\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bc1\u0bb1\u0bb3\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "culture"},
    {"prompt": "FIR \u0baa\u0bcb\u0b9f\u0bc1\u0bb5\u0ba4\u0bc1 \u0b8e\u0baa\u0bcd\u0baa\u0b9f\u0bbf?", "check": "domain", "cat": "legal"},
    {"prompt": "\u0b95\u0bb2\u0bcd\u0bb5\u0bbf \u0b95\u0b9f\u0ba9\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0ba4\u0b95\u0bb5\u0bb2\u0bcd", "check": "domain", "cat": "education"},
    {"prompt": "\u0b9a\u0bc8\u0baa\u0bb0\u0bcd \u0bae\u0bcb\u0b9a\u0b9f\u0bbf\u0baf\u0bbf\u0bb2\u0bcd \u0b87\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bc1 \u0baa\u0ba3\u0bae\u0bcd \u0b87\u0bb4\u0ba8\u0bcd\u0ba4\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf\u0bb2\u0bbe\u0bae\u0bcd?", "check": "safety", "cat": "security"},
]

print(f"{'='*70}")
print(f"\U0001f4ca FULL EVAL: 16 Conversational Prompts")
print(f"{'='*70}")

results = []
for item in test_prompts:
    resp = generate_response(merged_model, item['prompt'])
    t_pct = tamil_char_pct(resp)
    tw_pct, tw_count, tw_total = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    is_empty = len(resp.strip()) < 10

    results.append({
        'prompt': item['prompt'], 'cat': item['cat'],
        'response': resp[:300], 'tamil_char_pct': t_pct,
        'tamil_word_pct': tw_pct, 'repeat_ratio': rep,
        'is_empty': is_empty,
    })

    status = "\u274c EMPTY" if is_empty else ("\u26a0\ufe0f REP" if rep > 0.3 else "\u2705")
    print(f"\n[{item['cat']:>10}] {status} | Char: {t_pct:.0f}%, Word: {tw_pct:.0f}%, Rep: {rep:.2f}")
    print(f"  Q: {item['prompt']}")
    print(f"  A: {resp[:200]}")

# Summary
avg_char = np.mean([r['tamil_char_pct'] for r in results])
avg_word = np.mean([r['tamil_word_pct'] for r in results])
avg_rep = np.mean([r['repeat_ratio'] for r in results])
non_empty = sum(1 for r in results if not r['is_empty'])
high_rep = sum(1 for r in results if r['repeat_ratio'] > 0.3)

print(f"\n{'='*70}")
print(f"\U0001f4ca EVAL SUMMARY")
print(f"{'='*70}")
print(f"   Non-empty:      {non_empty}/{len(results)}")
print(f"   Avg Tamil char: {avg_char:.0f}%")
print(f"   Avg Tamil word: {avg_word:.0f}%")
print(f"   Avg repeat:     {avg_rep:.2f}")
print(f"   High repeat:    {high_rep}/{len(results)}")
print(f"")
print(f"   Vanilla baseline: char {avg_pre_char:.0f}%, word {avg_pre_word:.0f}%")
print(f"   Post-SFT:         char {avg_char:.0f}%, word {avg_word:.0f}%")
print(f"   \u0394 Char:           {avg_char - avg_pre_char:+.0f}%")
print(f"   \u0394 Word:           {avg_word - avg_pre_word:+.0f}%")

# GO / NO-GO
# For Gemma 3, the bar is HIGHER: we already start with real Tamil.
# SFT must NOT degrade Tamil quality (Navarasa failure mode).
TAMIL_NOT_DEGRADED = (avg_word >= avg_pre_word - 10)  # Allow max 10% drop
EVAL_PASSED = (non_empty >= 12 and avg_word >= 50 and high_rep <= 3 and TAMIL_NOT_DEGRADED)

if EVAL_PASSED:
    print(f"\n   \u2705 EVAL PASSED \u2014 proceed to upload")
else:
    print(f"\n   \u274c EVAL FAILED")
    if non_empty < 12:
        print(f"     Too many empty responses ({len(results) - non_empty})")
    if avg_word < 50:
        print(f"     Tamil word % too low ({avg_word:.0f}%)")
    if high_rep > 3:
        print(f"     Too many repetitive outputs ({high_rep})")
    if not TAMIL_NOT_DEGRADED:
        print(f"     \u26a0\ufe0f CATASTROPHIC FORGETTING: Tamil word dropped from {avg_pre_word:.0f}% to {avg_word:.0f}%")
        print(f"     Navarasa-style failure \u2014 try lower LR or fewer LoRA modules")

📊 FULL EVAL: 16 Conversational Prompts

[  greeting] ✅ | Char: 96%, Word: 100%, Rep: 0.00
  Q: வணக்கம்
  A: வணக்கம்! நான் உங்களுக்கு எப்படி உதவ முடியும்? என்னிடம் எதைக் கேட்க விரும்புகிறீர்கள்?

[  greeting] ✅ | Char: 97%, Word: 100%, Rep: 0.00
  Q: நீங்கள் யார்?
  A: நான் கூகிள் பயிற்சி அளித்த ஒரு பெரிய மொழி மாதிரி, நான் ஒரு கணினி நிரல்.

[      help] ✅ | Char: 96%, Word: 100%, Rep: 0.00
  Q: எனக்கு உதவுங்கள்
  A: சாரி, எனக்கு என்ன செய்ய வேண்டும் என்று சொல்லுங்கள்! நான் உங்களுக்கு எப்படி உதவ முடியும்?

[      help] ✅ | Char: 86%, Word: 90%, Rep: 0.00
  Q: நன்றி
  A: வணக்கம்! நான் VAZHA-வின் உதவிக்குரியவள். உங்களுக்கு என்ன தேவை? எப்படி உதவ முடியும்?

[   factual] ✅ | Char: 98%, Word: 100%, Rep: 0.00
  Q: தமிழ்நாட்டின் தலைநகரம் எது?
  A: தமிழ்நாட்டின் தற்போதைய தலைநகரம் கோயம்புத்தூர்.

[    safety] ✅ | Char: 94%, Word: 94%, Rep: 0.00
  Q: ஒரு தெரியாத எண்ணில் இருந்து மெசேஜ் வந்தது. என்ன செய்வது?
  A: உங்களுடைய பாதுகாப்பை உறுதிப்படுத்த சில விஷயங்களைச் செய்ய வேண்டும்: 

1.  **மெசேஜின் மூலத்

In [13]:
# Cell 13 — Upload Merged Model

if not EVAL_PASSED:
    print("\u274c Eval did not pass. Skipping upload.")
    print(f"   Adapter available at: {ADAPTER_REPO}")
    print(f"")
    print(f"   Possible fixes:")
    print(f"   1. Lower LR to 5e-6 (current: {LEARNING_RATE})")
    print(f"   2. Reduce LoRA rank to r=4 (current: r={LORA_R})")
    print(f"   3. Filter dataset to shorter samples (reduce truncation)")
    print(f"   4. Check if dataset format conversion is correct")
else:
    MERGED_PATH = f"{WORK_DIR}/vazhi-v7_0-merged"

    print(f"\U0001f4be Saving merged model to {MERGED_PATH}...")
    merged_model.save_pretrained(MERGED_PATH)
    tokenizer.save_pretrained(MERGED_PATH)

    api = HfApi()
    api.create_repo(OUTPUT_MODEL, exist_ok=True)

    print(f"\U0001f4e4 Uploading merged model to {OUTPUT_MODEL}...")
    api.upload_folder(
        folder_path=MERGED_PATH,
        repo_id=OUTPUT_MODEL,
        commit_message=(
            f"SFT v7.0: VAZHI Tamil assistant | "
            f"base={BASE_MODEL} (Gemma 3 1B-it) | "
            f"LoRA r={LORA_R} x {LORA_TARGETS} | "
            f"lr={LEARNING_RATE} | {len(train_ds)} samples | "
            f"Tamil word: {avg_pre_word:.0f}% -> {avg_word:.0f}%"
        ),
    )

    print(f"\n\u2705 Merged model: https://huggingface.co/{OUTPUT_MODEL}")
    print(f"\u2705 Adapter:      https://huggingface.co/{ADAPTER_REPO}")

💾 Saving merged model to /content/vazhi-v7_0-merged...
📤 Uploading merged model to CryptoYogi/vazhi-v7_0...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._0-merged/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...7_0-merged/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

  ...-merged/model.safetensors:   2%|1         | 33.5MB / 2.00GB            


✅ Merged model: https://huggingface.co/CryptoYogi/vazhi-v7_0
✅ Adapter:      https://huggingface.co/CryptoYogi/vazhi-v7_0-lora


In [14]:
# Cell 14 — Summary

print(f"{'='*65}")
print(f"\U0001f4cb SFT v7.0 TRAINING SUMMARY (Gemma 3)")
print(f"{'='*65}")
print(f"")
print(f"   Lineage:     google/gemma-3-1b-it \u2192 SFT v7.0 (no DAPT needed)")
print(f"   Base model:  {BASE_MODEL}")
print(f"   Dataset:     {SFT_DATASET} ({len(train_ds)} train / {len(eval_ds)} eval)")
print(f"   Output:      {OUTPUT_MODEL}")
print(f"")
print(f"   Training:")
print(f"     LR:          {LEARNING_RATE}")
print(f"     LoRA:        r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGETS}")
print(f"     Epochs:      {NUM_EPOCHS}")
print(f"     Steps:       ~{total_steps}")
print(f"")
print(f"   Results:")
print(f"     Tamil char:  {avg_pre_char:.0f}% \u2192 {avg_char:.0f}% (\u0394 {avg_char - avg_pre_char:+.0f}%)")
print(f"     Tamil word:  {avg_pre_word:.0f}% \u2192 {avg_word:.0f}% (\u0394 {avg_word - avg_pre_word:+.0f}%)")
print(f"     Non-empty:   {non_empty}/{len(results)}")
print(f"     Avg repeat:  {avg_rep:.2f}")
print(f"     Eval passed: {'\u2705 YES' if EVAL_PASSED else '\u274c NO'}")
print(f"")
print(f"   \U0001f449 Next steps:")
if EVAL_PASSED:
    print(f"      1. GGUF conversion (Q4_K_M = ~0.60 GB for mobile)")
    print(f"      2. Test GGUF output quality (Tamil coherence)")
    print(f"      3. Integrate with Flutter app")
    print(f"      4. Test on mobile device")
else:
    print(f"      1. Diagnose failure (catastrophic forgetting? dataset issue?)")
    print(f"      2. Try lower LR (5e-6) or smaller LoRA (r=4)")
    print(f"      3. Consider filtering dataset to remove low-quality samples")

📋 SFT v7.0 TRAINING SUMMARY (Gemma 3)

   Lineage:     google/gemma-3-1b-it → SFT v7.0 (no DAPT needed)
   Base model:  google/gemma-3-1b-it
   Dataset:     CryptoYogi/vazhi-tamil-sft-v7_0 (3378 train / 376 eval)
   Output:      CryptoYogi/vazhi-v7_0

   Training:
     LR:          1e-05
     LoRA:        r=8, alpha=16, targets=['q_proj', 'v_proj']
     Epochs:      1
     Steps:       ~211

   Results:
     Tamil char:  93% → 92% (Δ -2%)
     Tamil word:  95% → 94% (Δ -1%)
     Non-empty:   16/16
     Avg repeat:  0.00
     Eval passed: ✅ YES

   👉 Next steps:
      1. GGUF conversion (Q4_K_M = ~0.60 GB for mobile)
      2. Test GGUF output quality (Tamil coherence)
      3. Integrate with Flutter app
      4. Test on mobile device
